In [12]:
# Preparation. Set USE_LOCAL_EXPORT=True to read a folder from
# scripts/export_labeling_run.py; False uses live Databricks MLflow.
# SOURCE_RUN_ID is only needed the first time an episode is opened
# (when gold has no "pending" yet).
import json
from pathlib import Path

from drzavni_posao import MediaStore, MlflowTracker, Stage
from drzavni_posao.transcript import Transcript

USE_LOCAL_EXPORT = True
SOURCE_RUN_ID = "1c519fba673841fa9188addf9a720ae6"
EPISODE_KEY = "1"
REDO = []  # Zero-based indexes in gold["segments"].
CHARACTERS = ["Djordje Cvarkov", "Dragan Torbica", "Boskic"]

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CATALOG_PATH = REPO_ROOT / "data" / "youtube_episodes.json"
if USE_LOCAL_EXPORT:
    AUDIO_PATHS = REPO_ROOT / "downloads"
    LABEL_DIR = REPO_ROOT / "data" / "labels"
    RUN_EXPORT = REPO_ROOT / "data" / "mlflow_exports" / SOURCE_RUN_ID
    EXPERIMENT_NAME = None
else:
    AUDIO_PATHS = Path("/Volumes/luka_erceg/drzavni_posao/audios")
    LABEL_DIR = Path(
        "/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/labels"
    )
    RUN_EXPORT = None
    EXPERIMENT_NAME = (
        "/Users/luka.erceg@databricks.com/drzavni_posao/transcription"
    )

tracker = None if USE_LOCAL_EXPORT else MlflowTracker(EXPERIMENT_NAME)
label_path = LABEL_DIR / f"{EPISODE_KEY}.json"
gold = (
    json.loads(label_path.read_text(encoding="utf-8"))
    if label_path.exists()
    else None
)
needs_source = gold is None or "pending" not in gold


def stage_transcript(stage: Stage) -> Transcript:
    if USE_LOCAL_EXPORT:
        path = (
            RUN_EXPORT
            / "episodes"
            / EPISODE_KEY
            / stage.value
            / "transcript.json"
        )
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing {path}. Export the run or pick an episode that has "
                f"{stage.value}."
            )
        return Transcript.from_dict(
            json.loads(path.read_text(encoding="utf-8"))
        )
    source = tracker.episode_run_states(SOURCE_RUN_ID)[EPISODE_KEY]
    return tracker.stage_transcript(source.run_id, stage)


def load_proposal():
    if USE_LOCAL_EXPORT:
        config_path = RUN_EXPORT / "config.json"
        if not config_path.is_file():
            raise FileNotFoundError(
                f"Missing export at {RUN_EXPORT}. Run "
                f"scripts/export_labeling_run.py --run-id {SOURCE_RUN_ID}"
            )
        config = json.loads(config_path.read_text(encoding="utf-8"))
    else:
        config = tracker._configuration(SOURCE_RUN_ID)
    return {key: config[key] for key in ("audio", "transcription")}


if needs_source:
    proposal = load_proposal()
    if gold is not None and gold["proposal"] != proposal:
        raise ValueError(
            "This episode was labeled from a different ASR configuration"
        )
else:
    proposal = gold["proposal"]

media = MediaStore(
    AUDIO_PATHS,
    LABEL_DIR,
    catalog_path=CATALOG_PATH,
    **proposal["audio"],
)
episode = media.episode(EPISODE_KEY)
trim_start = proposal["audio"]["trim_start"]


def overlap(segment, segment_trim, label):
    return max(
        0,
        min(segment["end"] + segment_trim, label["end"])
        - max(segment["start"] + segment_trim, label["start"]),
    )


if needs_source:
    episode.transcript = stage_transcript(Stage.TRANSCRIPTION)
    if gold is None:
        gold = {
            "schema_version": 1,
            "episode_key": EPISODE_KEY,
            "timebase": "raw_audio",
            "proposal": proposal,
            "segments": [],
        }
    gold["pending"] = [
        segment
        for segment in episode.transcript.segments
        if not any(
            overlap(segment, trim_start, label) for label in gold["segments"]
        )
    ]

redo_segments = []
for index in dict.fromkeys(REDO):
    if not isinstance(index, int) or not 0 <= index < len(gold["segments"]):
        raise IndexError(f"Invalid REDO index {index!r}")
    label = gold["segments"][index]
    redo_segments.append(
        {
            "start": label["start"] - trim_start,
            "end": label["end"] - trim_start,
            "text": f"Current label: {label['speaker']}",
            "_redo_index": index,
        }
    )

for speaker, candidates in gold.pop("gallery_candidates", {}).items():
    for segment in gold["segments"]:
        if segment["speaker"] == speaker and any(
            segment["start"] == candidate["start"]
            and segment["end"] == candidate["end"]
            for candidate in candidates
        ):
            segment["gallery_candidate"] = True


def save():
    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text(
        json.dumps(gold, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )


def gold_speaker(segment, candidate_trim_start=0):
    matches = [
        (overlap(segment, candidate_trim_start, label), label["speaker"])
        for label in gold["segments"]
    ]
    seconds, speaker = max(
        matches, key=lambda match: match[0], default=(0, None)
    )
    return speaker if seconds else None


save()
audio = episode.audio()
print(
    f"{len(gold['segments'])} labeled, {len(gold['pending'])} pending, "
    f"{sum(segment.get('gallery_candidate', False) for segment in gold['segments'])} "
    "gallery candidates"
)
print(f"mode={'local_export' if USE_LOCAL_EXPORT else 'mlflow'}")
print(f"audio={episode.audio_path}")
print(f"labels={label_path}")
if needs_source:
    print(f"source={RUN_EXPORT if USE_LOCAL_EXPORT else SOURCE_RUN_ID}")
else:
    print("source=existing labels")


0 labeled, 117 pending, 0 gallery candidates
mode=local_export
audio=/Users/luka.erceg/drzavni_posao/downloads/season-1/DRŽAVNI POSAO [HQ] - Ep.1： Rakija (24.09.2012.).mp3
labels=/Users/luka.erceg/drzavni_posao/data/labels/1.json
source=/Users/luka.erceg/drzavni_posao/data/mlflow_exports/1c519fba673841fa9188addf9a720ae6


In [ ]:
# Labeling UI. Toggle ✓ before choosing a character to save a gallery candidate.
import ipywidgets as widgets
from IPython.display import Audio, clear_output, display

from drzavni_posao.constants import ANONYMOUS_LABEL, SR

queue = redo_segments + gold["pending"]
skipped = []
position = 0
gallery_selected = False
output = widgets.Output()
gallery = widgets.Button(description="✓", tooltip="Gallery candidate")
buttons = [widgets.Button(description=name) for name in [*CHARACTERS, ANONYMOUS_LABEL]]
skip = widgets.Button(description="Skip")
controls = [gallery, *buttons, skip]


def toggle_gallery(_):
    global gallery_selected
    gallery_selected = not gallery_selected
    gallery.description = "✓ ON" if gallery_selected else "✓"


def show():
    with output:
        clear_output(wait=True)
        if position >= len(queue):
            print(
                "Pass complete. Rerun the preparation cell to revisit skipped segments."
            )
            for button in controls:
                button.disabled = True
            return
        segment = queue[position]
        print(
            f"{position + 1}/{len(queue)}  "
            f"{segment['start']:.1f}–{segment['end']:.1f}s"
        )
        print(segment.get("text", "").strip())
        start = max(0, int((segment["start"] - 0.05) * SR))
        end = int((segment["end"] + 0.05) * SR)
        display(Audio(audio[start:end], rate=SR, autoplay=True))


def advance(speaker=None):
    global position, gallery_selected
    segment = queue[position]
    redo_index = segment.get("_redo_index")
    if speaker is None:
        if redo_index is None:
            skipped.append(segment)
    else:
        label = {
            "start": segment["start"] + trim_start,
            "end": segment["end"] + trim_start,
            "speaker": speaker,
        }
        if gallery_selected and speaker != ANONYMOUS_LABEL:
            label["gallery_candidate"] = True
        if redo_index is None:
            gold["segments"].append(label)
        else:
            gold["segments"][redo_index] = label
    gallery_selected = False
    gallery.description = "✓"
    position += 1
    gold["pending"] = [
        item
        for item in queue[position:] + skipped
        if "_redo_index" not in item
    ]
    save()
    show()


gallery.on_click(toggle_gallery)
for button in buttons:
    button.on_click(lambda _, speaker=button.description: advance(speaker))
skip.on_click(lambda _: advance())
display(widgets.VBox([output, widgets.HBox(controls)]))
show()


In [7]:
# Score character assignment against this episode's gold labels.
# Local export: uses character_assigner/ from RUN_EXPORT (ASSIGNMENT_RUN_ID
# defaults to SOURCE_RUN_ID). Live MLflow: set ASSIGNMENT_RUN_ID to any finished
# assignment parent run.
import pandas as pd
from IPython.display import display

ASSIGNMENT_RUN_ID = SOURCE_RUN_ID if USE_LOCAL_EXPORT else ""
if not ASSIGNMENT_RUN_ID:
    raise ValueError("Set ASSIGNMENT_RUN_ID to a finished assignment parent run")

if USE_LOCAL_EXPORT:
    if ASSIGNMENT_RUN_ID != SOURCE_RUN_ID:
        raise ValueError(
            "Local export mode scores the exported SOURCE_RUN_ID only; "
            f"got ASSIGNMENT_RUN_ID={ASSIGNMENT_RUN_ID!r}"
        )
    assignment_config = json.loads(
        (RUN_EXPORT / "config.json").read_text(encoding="utf-8")
    )
    episode.transcript = stage_transcript(Stage.CHARACTER_ASSIGNER)
else:
    if tracker is None:
        tracker = MlflowTracker(EXPERIMENT_NAME)
    assignment_config = tracker._configuration(ASSIGNMENT_RUN_ID)
    assignment = tracker.episode_run_states(ASSIGNMENT_RUN_ID)[EPISODE_KEY]
    tracker.restore_stage(episode, assignment.run_id, Stage.CHARACTER_ASSIGNER)

assignment_trim_start = assignment_config["audio"]["trim_start"]

rows = []
for segment in episode.transcript.segments:
    expected = gold_speaker(segment, assignment_trim_start)
    if expected is not None:
        rows.append(
            {
                "start": segment["start"],
                "end": segment["end"],
                "gold": expected,
                "assigned": segment.get("speaker") or "[unassigned]",
                "text": segment.get("text", "").strip(),
            }
        )

results = pd.DataFrame(rows)
if results.empty:
    print("No assignment segments overlap the gold labels.")
else:
    print(f"accuracy: {(results['gold'] == results['assigned']).mean():.1%}")
    display(pd.crosstab(results["gold"], results["assigned"]))
    display(results[results["gold"] != results["assigned"]])


NameError: name 'episode' is not defined

In [ ]:
# Build the expanded speaker gallery from every labeled episode.
gallery_path = Path("data/speaker_galleries/expanded.json")
gallery = {}

for path in sorted(Path(LABEL_DIR).glob("[0-9]*.json")):
    labels = json.loads(path.read_text(encoding="utf-8"))
    for segment in labels.get("segments", []):
        if segment.get("gallery_candidate"):
            gallery.setdefault(segment["speaker"], []).append(
                {
                    "episode_key": labels["episode_key"],
                    "start": segment["start"],
                    "end": segment["end"],
                }
            )

for clips in gallery.values():
    clips.sort(key=lambda clip: (clip["episode_key"], clip["start"]))

gallery_path.parent.mkdir(parents=True, exist_ok=True)
gallery_path.write_text(
    json.dumps(gallery, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(
    f"Saved {sum(map(len, gallery.values()))} clips for {len(gallery)} characters "
    f"to {gallery_path}"
)
gallery
